# 🐾 SuperVet IA: aplicação completa no Colab (Sprint 4)

Aplicação única, sem backend externo:

```
Interface (Gradio)  ⇄  Banco de dados (SQLite)  ⇄  IA: motor de regras + LLM (Ollama)
```

**Abordagem híbrida (Sprint 3):** o **motor de regras decide e justifica** a prioridade; o **LLM só redige** o texto.

| Requisito da entrega | Onde está |
|---|---|
| IA definida no Sprint 3 (regras + IA Generativa) | Seções 3 e 4 |
| Dados preparados, testes e validação | Seções 2 e 5 |
| Integração interface ↔ banco ↔ serviço de IA | Seções 2, 4 e 7 (Gradio + SQLite + Ollama) |
| Acessível ao usuário final (tutor, veterinário, clínica) | Seção 7: app com 3 abas |
| Evidências e avaliação dos resultados | Seção 6 |

**Como rodar:** `Ambiente de execução → Alterar tipo → GPU T4` e depois `Executar tudo`. O link do app aparece na última célula.
Sem GPU, troque `MODELO` por `llama3.2:3b` na Seção 1.

## 1. Ambiente: Ollama + Gradio

In [ ]:
!nvidia-smi -L || echo "Sem GPU detectada: use MODELO = llama3.2:3b"
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install gradio

In [ ]:
import requests, subprocess, time

MODELO = "qwen2.5:7b"          # alternativas: "llama3.1:8b" | "llama3.2:3b" (sem GPU)
OLLAMA_URL = "http://127.0.0.1:11434"

def ollama_ativo():
    try:
        return requests.get(OLLAMA_URL, timeout=2).status_code == 200
    except Exception:
        return False

In [ ]:
# Sobe o servidor do Ollama em segundo plano e baixa o modelo
if not ollama_ativo():
    subprocess.Popen(["ollama", "serve"], stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(30):
        if ollama_ativo(): break
        time.sleep(1)

!ollama pull {MODELO}

r = requests.post(f"{OLLAMA_URL}/api/chat", timeout=300, json={"model": MODELO, "stream": False,
    "messages": [{"role": "user", "content": "Responda só: OK, SuperVet pronto."}]})
print("Ollama ativo:", ollama_ativo(), "|", r.json()["message"]["content"])

## 2. Banco de dados (SQLite) com dados de teste
Mesmos nomes de tabelas e colunas do documento do Sprint 3. As datas são **relativas a hoje**, então os cenários valem em qualquer dia.
4 pets em situações diferentes: saudável, vacina e retorno atrasados, perda de peso com sintoma recorrente e anomalia cardíaca com alerta da coleira.

In [ ]:
import os, sqlite3, random, json, re, unicodedata
from contextlib import closing
from collections import Counter
from dataclasses import dataclass, asdict
from datetime import datetime, date, timedelta

DB_PATH = "supervet.db"
AGORA = datetime.now().replace(microsecond=0)
HOJE = AGORA.date()
def passado(dias): return (HOJE - timedelta(days=dias)).isoformat()
def futuro(dias):  return (HOJE + timedelta(days=dias)).isoformat()

SCHEMA = """
CREATE TABLE TB_ESPECIE (ID_ESPECIE INTEGER PRIMARY KEY, NOME TEXT);
CREATE TABLE TB_RACA (ID_RACA INTEGER PRIMARY KEY, NOME TEXT, PORTE TEXT, ID_ESPECIE INTEGER);
CREATE TABLE TB_TUTOR (ID_TUTOR INTEGER PRIMARY KEY, NOME TEXT, CPF TEXT, EMAIL TEXT, TELEFONE TEXT, CONSENTE_IA INTEGER DEFAULT 1);
CREATE TABLE TB_PET (ID_PET INTEGER PRIMARY KEY, NOME TEXT, SEXO TEXT, DATA_NASCIMENTO TEXT, ID_RACA INTEGER, ID_TUTOR INTEGER);
CREATE TABLE TB_PRONTUARIO (ID_PRONTUARIO INTEGER PRIMARY KEY AUTOINCREMENT, ID_PET INTEGER, DATA TEXT, DIAGNOSTICO TEXT, SINTOMAS TEXT,
                            PESO REAL, TEMPERATURA REAL, EXAMES_SOLICITADOS TEXT, DATA_RETORNO TEXT);
CREATE TABLE TB_CONSULTA (ID_CONSULTA INTEGER PRIMARY KEY AUTOINCREMENT, ID_PET INTEGER, DATA TEXT, STATUS TEXT, MODALIDADE TEXT);
CREATE TABLE TB_HISTORICO_SAUDE (ID_PET INTEGER, EVENTO TEXT, DATA_EVENTO TEXT, DETALHES TEXT);
CREATE TABLE TB_PRESCRICAO (ID_PET INTEGER, TIPO TEXT, DESCRICAO TEXT, POSOLOGIA TEXT, ATIVA INTEGER);
CREATE TABLE TB_COLEIRA (ID_PET INTEGER, DATA_HORA TEXT, LATITUDE REAL, LONGITUDE REAL, BPM INTEGER, ALERTA TEXT);
CREATE TABLE TB_NOTIFICACAO (ID_NOTIFICACAO INTEGER PRIMARY KEY AUTOINCREMENT, ID_PET INTEGER, MENSAGEM TEXT, PRIORIDADE TEXT,
                             STATUS TEXT DEFAULT 'PENDENTE', CRIADA_EM TEXT);
"""

def q(sql, *args):
    with closing(sqlite3.connect(DB_PATH)) as con:
        con.row_factory = sqlite3.Row
        return [dict(r) for r in con.execute(sql, args)]

def exec_sql(sql, *args):
    with closing(sqlite3.connect(DB_PATH)) as con:
        cur = con.execute(sql, args); con.commit()
        return cur.rowcount

def _coleira(id_pet, normal, anomala=None, n_anomalas=0, n_alertas=0, seed=0):
    # Uma leitura a cada 30 min nas últimas 24h; as últimas `n_anomalas` ficam fora da faixa
    rnd = random.Random(seed)
    linhas = []
    for i in range(48, 0, -1):
        linhas.append([id_pet, (AGORA - timedelta(minutes=30 * i)).isoformat(),
                       -23.55 + rnd.uniform(-0.001, 0.001), -46.63 + rnd.uniform(-0.001, 0.001), rnd.randint(*normal), None])
    for k in range(1, n_anomalas + 1): linhas[-k][4] = rnd.randint(*anomala)
    for k in range(1, n_alertas + 1):  linhas[-k][5] = "TAQUICARDIA"
    return linhas

def init_db():
    if os.path.exists(DB_PATH): os.remove(DB_PATH)
    with closing(sqlite3.connect(DB_PATH)) as con:
        con.executescript(SCHEMA)
        con.executemany("INSERT INTO TB_ESPECIE VALUES (?,?)", [(1, "Cachorro"), (2, "Gato")])
        con.executemany("INSERT INTO TB_RACA VALUES (?,?,?,?)", [(1, "Labrador", "GRANDE", 1), (2, "Shih Tzu", "PEQUENO", 1),
                                                                 (3, "SRD", "MEDIO", 2), (4, "Pastor Alemão", "GRANDE", 1)])
        # Dados pessoais fictícios: existem só para provar que NÃO saem do banco para a IA
        con.executemany("INSERT INTO TB_TUTOR VALUES (?,?,?,?,?,1)",
                        [(i, f"Tutor Exemplo {i}", f"000.000.00{i}-0{i}", f"tutor{i}@exemplo.com", f"(11) 90000-000{i}") for i in range(1, 5)])
        con.executemany("INSERT INTO TB_PET VALUES (?,?,?,?,?,?)", [
            (1, "Thor", "M", passado(5 * 365),  1, 1), (2, "Mel", "F", passado(9 * 365),  2, 2),
            (3, "Luna", "F", passado(12 * 365), 3, 3), (4, "Rex", "M", passado(7 * 365),  4, 4)])
        con.executemany("""INSERT INTO TB_PRONTUARIO (ID_PET,DATA,DIAGNOSTICO,SINTOMAS,PESO,TEMPERATURA,EXAMES_SOLICITADOS,DATA_RETORNO)
                           VALUES (?,?,?,?,?,?,?,?)""", [
            (1, passado(200), "Check-up anual", "", 31.8, 38.4, "", None),
            (1, passado(45),  "Check-up de rotina, sem alterações", "", 32.0, 38.5, "", futuro(45)),
            (2, passado(180), "Avaliação geral", "", 7.1, 38.6, "", None),
            (2, passado(50),  "Tártaro dentário grau II", "mau hálito", 7.2, 38.7, "Limpeza dentária (agendar)", passado(20)),
            (3, passado(70),  "Episódio de vômito isolado", "vômito", 4.8, 38.4, "", None),
            (3, passado(24),  "Doença renal crônica em investigação", "vômito, apatia", 4.2, 38.3, "ureia, creatinina", passado(10)),
            (4, passado(120), "Avaliação geral", "", 36.0, 38.6, "", None),
            (4, passado(10),  "Tosse persistente em investigação", "tosse, cansaço", 35.5, 38.8, "raio-x de tórax, ecocardiograma", passado(3))])
        con.executemany("INSERT INTO TB_CONSULTA (ID_PET,DATA,STATUS,MODALIDADE) VALUES (?,?,?,'PRESENCIAL')", [
            (1, passado(45), "REALIZADA"), (1, passado(200), "REALIZADA"), (2, passado(50), "REALIZADA"), (2, passado(180), "REALIZADA"),
            (3, passado(24), "REALIZADA"), (3, passado(70), "REALIZADA"),
            (4, passado(10), "REALIZADA"), (4, passado(120), "REALIZADA"), (4, passado(3), "FALTOU")])
        con.executemany("INSERT INTO TB_HISTORICO_SAUDE VALUES (?,?,?,?)", [
            (1, "VACINA", passado(120), "V10 e antirrábica"), (2, "VACINA", passado(420), "V10"),
            (3, "VACINA", passado(200), "V4 felina"),         (4, "VACINA", passado(100), "V10")])
        con.executemany("INSERT INTO TB_PRESCRICAO VALUES (?,?,?,?,1)", [
            (2, "DIETA", "Ração para saúde bucal", "Conforme embalagem"),
            (3, "DIETA", "Ração renal", "Conforme orientação do veterinário"),
            (3, "MEDICAMENTO", "Protetor gástrico", "Conforme receita do veterinário"),
            (4, "RESTRICAO", "Evitar exercício intenso até a reavaliação", "")])
        leituras = (_coleira(1, (65, 95), seed=1) + _coleira(2, (105, 135), seed=2) + _coleira(3, (150, 190), seed=3) +
                    _coleira(4, (70, 95), anomala=(115, 140), n_anomalas=5, n_alertas=2, seed=4))
        con.executemany("INSERT INTO TB_COLEIRA VALUES (?,?,?,?,?,?)", leituras)
        con.commit()

init_db()
print({t: q(f"SELECT COUNT(*) AS n FROM {t}")[0]["n"] for t in ["TB_PET", "TB_PRONTUARIO", "TB_CONSULTA", "TB_COLEIRA", "TB_NOTIFICACAO"]})

## 3. Motor de regras: contexto sem dados pessoais + score de prioridade
O `montar_contexto` lê o banco e **seleciona só as colunas necessárias** (sem CPF, e-mail, telefone, nome do tutor nem GPS).
Cada regra devolve **pontos + motivo**; o score é a soma (máx. 100), logo é auditável por um veterinário.

> ⚠️ Faixas de BPM, intervalo de vacina e serviços sugeridos são **referências ilustrativas** e devem ser validados por um veterinário. A coleira não informa atividade física, então leituras durante exercício podem gerar falso alerta.

In [ ]:
def _norm(s):
    s = unicodedata.normalize("NFD", str(s or ""))
    return "".join(c for c in s if unicodedata.category(c) != "Mn").lower().strip()

def montar_contexto(id_pet, agora=None):
    agora = agora or datetime.now().replace(microsecond=0)
    pet = q("""SELECT p.NOME, p.SEXO, p.DATA_NASCIMENTO, r.NOME AS RACA, r.PORTE, e.NOME AS ESPECIE
               FROM TB_PET p JOIN TB_RACA r ON r.ID_RACA = p.ID_RACA JOIN TB_ESPECIE e ON e.ID_ESPECIE = r.ID_ESPECIE
               WHERE p.ID_PET = ?""", id_pet)[0]
    corte = (agora - timedelta(hours=24)).isoformat()
    return {
        "id_pet": id_pet, "referencia_em": agora.isoformat(),
        "pet": {"nome": pet["NOME"], "sexo": pet["SEXO"], "especie": pet["ESPECIE"], "raca": pet["RACA"], "porte": pet["PORTE"],
                "idade_anos": (agora.date() - date.fromisoformat(pet["DATA_NASCIMENTO"])).days // 365},
        "prontuarios": q("""SELECT DATA AS data, DIAGNOSTICO AS diagnostico, SINTOMAS AS sintomas, PESO AS peso_kg, TEMPERATURA AS temperatura_c,
                            EXAMES_SOLICITADOS AS exames_solicitados, DATA_RETORNO AS data_retorno
                            FROM TB_PRONTUARIO WHERE ID_PET = ? ORDER BY DATA""", id_pet),
        "consultas": q("SELECT DATA AS data, STATUS AS status FROM TB_CONSULTA WHERE ID_PET = ? ORDER BY DATA", id_pet),
        "historico_saude": q("SELECT EVENTO AS evento, DATA_EVENTO AS data, DETALHES AS detalhes FROM TB_HISTORICO_SAUDE WHERE ID_PET = ? ORDER BY DATA_EVENTO", id_pet),
        "prescricoes": [dict(p, ativa=bool(p["ativa"])) for p in
                        q("SELECT TIPO AS tipo, DESCRICAO AS descricao, POSOLOGIA AS posologia, ATIVA AS ativa FROM TB_PRESCRICAO WHERE ID_PET = ?", id_pet)],
        "coleira_24h": q("SELECT DATA_HORA AS datahora, BPM AS bpm, ALERTA AS alerta FROM TB_COLEIRA WHERE ID_PET = ? AND DATA_HORA >= ? ORDER BY DATA_HORA", id_pet, corte),
    }

LIMIARES = {
    "retorno_atraso_pts": [(14, 15), (60, 25), (10**6, 30)],   # (até N dias de atraso, pontos)
    "protocolo_vacinal_dias": 365, "vacina_pts": 20,
    "peso_pct": [(10, 20), (5, 10)],                            # (variação >= %, pontos)
    "sintomas_recentes_dias": 30, "sintomas_recentes_pts": 10,
    "sintoma_recorrente_janela_dias": 180, "sintoma_recorrente_pts": 10,
    "bpm_persistente_min_leituras": 3, "bpm_persistente_pts": 25, "bpm_isolado_pts": 10,
    "alerta_coleira_pts": 20, "consulta_perdida_dias": 90, "consulta_perdida_pts": 10,
    "niveis": [(75, "CRITICO"), (50, "ALTO"), (25, "MEDIO"), (0, "BAIXO")],
}

def faixa_bpm(especie, porte):
    if _norm(especie) == "gato": return (140, 220)
    return {"pequeno": (100, 140), "medio": (80, 120), "grande": (60, 100)}.get(_norm(porte), (60, 140))

@dataclass
class Achado:
    codigo: str
    pontos: int
    motivo: str

def _d(s): return date.fromisoformat(str(s)[:10])
def _br(d): return d.strftime("%d/%m/%Y")

def regra_retorno(ctx, hoje):
    if not ctx["prontuarios"]: return []
    ult = max(ctx["prontuarios"], key=lambda p: p["data"])
    if not ult.get("data_retorno"): return []
    ret = _d(ult["data_retorno"])
    if ret >= hoje: return []
    if any(_norm(c["status"]) in {"realizada", "concluida"} and _d(c["data"]) > _d(ult["data"]) for c in ctx["consultas"]): return []
    atraso = (hoje - ret).days
    pts = next(p for lim, p in LIMIARES["retorno_atraso_pts"] if atraso <= lim)
    return [Achado("RETORNO_ATRASADO", pts, f"O retorno previsto para {_br(ret)} está atrasado há {atraso} dias.")]

def regra_vacina(ctx, hoje):
    vac = [e for e in ctx["historico_saude"] if "vacina" in _norm(e["evento"])]
    if not vac: return [Achado("VACINA_SEM_REGISTRO", LIMIARES["vacina_pts"], "Não há registro de vacinação no histórico.")]
    ult = max(_d(e["data"]) for e in vac); dias = (hoje - ult).days
    if dias > LIMIARES["protocolo_vacinal_dias"]:
        return [Achado("VACINA_ATRASADA", LIMIARES["vacina_pts"],
                       f"A última vacinação foi em {_br(ult)} (há {dias} dias), acima do intervalo de referência de {LIMIARES['protocolo_vacinal_dias']} dias.")]
    return []

def regra_peso(ctx, hoje):
    pesos = [p for p in sorted(ctx["prontuarios"], key=lambda p: p["data"]) if p.get("peso_kg")]
    if len(pesos) < 2: return []
    ant, atual = pesos[-2]["peso_kg"], pesos[-1]["peso_kg"]
    var = (atual - ant) / ant * 100
    for lim, pts in LIMIARES["peso_pct"]:
        if abs(var) >= lim:
            return [Achado("VARIACAO_PESO", pts, f"O peso {'caiu' if var < 0 else 'subiu'} {abs(var):.1f}% entre os dois últimos prontuários ({ant:.1f} kg → {atual:.1f} kg).")]
    return []

def _tokens(s): return [t.strip() for t in re.split(r"[,;]", (s or "").lower()) if t.strip()]

def regra_sintomas(ctx, hoje):
    achados = []
    rec = [p for p in ctx["prontuarios"] if p.get("sintomas") and (hoje - _d(p["data"])).days <= LIMIARES["sintomas_recentes_dias"]]
    if rec:
        lista = ", ".join(sorted({t for p in rec for t in _tokens(p["sintomas"])}))
        achados.append(Achado("SINTOMAS_RECENTES", LIMIARES["sintomas_recentes_pts"], f"Sintomas registrados nos últimos {LIMIARES['sintomas_recentes_dias']} dias: {lista}."))
    cont, exib = Counter(), {}
    for p in ctx["prontuarios"]:
        if (hoje - _d(p["data"])).days <= LIMIARES["sintoma_recorrente_janela_dias"]:
            for t in set(_tokens(p["sintomas"])):
                cont[_norm(t)] += 1; exib[_norm(t)] = t
    recorrentes = [exib[k] for k, c in cont.items() if c >= 2]
    if recorrentes:
        achados.append(Achado("SINTOMA_RECORRENTE", LIMIARES["sintoma_recorrente_pts"], f"Sintoma recorrente em mais de um prontuário: {', '.join(recorrentes)}."))
    return achados

def regra_bpm(ctx, hoje):
    lo, hi = faixa_bpm(ctx["pet"]["especie"], ctx["pet"]["porte"])
    fora = [l["bpm"] for l in ctx["coleira_24h"] if l.get("bpm") is not None and not (lo <= l["bpm"] <= hi)]
    if not fora: return []
    persistente = len(fora) >= LIMIARES["bpm_persistente_min_leituras"]
    return [Achado("BPM_PERSISTENTE" if persistente else "BPM_ISOLADO",
                   LIMIARES["bpm_persistente_pts"] if persistente else LIMIARES["bpm_isolado_pts"],
                   f"{len(fora)} leitura(s) de batimento fora da faixa de referência ({lo}–{hi} bpm) nas últimas 24h, com valores entre {min(fora)} e {max(fora)} bpm.")]

def regra_alerta_coleira(ctx, hoje):
    alertas = [l["alerta"] for l in ctx["coleira_24h"] if l.get("alerta")]
    if not alertas: return []
    return [Achado("ALERTA_COLEIRA", LIMIARES["alerta_coleira_pts"], f"A coleira emitiu {len(alertas)} alerta(s) nas últimas 24h (tipo: {', '.join(sorted(set(alertas))).lower()}).")]

def regra_consulta_perdida(ctx, hoje):
    perdidas = [c for c in ctx["consultas"] if _norm(c["status"]) in {"faltou", "nao_compareceu"} and (hoje - _d(c["data"])).days <= LIMIARES["consulta_perdida_dias"]]
    if not perdidas: return []
    return [Achado("CONSULTA_PERDIDA", LIMIARES["consulta_perdida_pts"], f"O tutor não compareceu à consulta de {_br(max(_d(c['data']) for c in perdidas))}.")]

REGRAS = [regra_retorno, regra_vacina, regra_peso, regra_sintomas, regra_bpm, regra_alerta_coleira, regra_consulta_perdida]

def nivel_de(score): return next(n for lim, n in LIMIARES["niveis"] if score >= lim)

def calcular_score(ctx):
    hoje = datetime.fromisoformat(ctx["referencia_em"]).date()
    achados = sorted((a for regra in REGRAS for a in regra(ctx, hoje)), key=lambda a: -a.pontos)
    score = min(100, sum(a.pontos for a in achados))
    return {"score": score, "nivel": nivel_de(score), "achados": [asdict(a) for a in achados]}

# ── Recomendação de serviços (determinística, referência ilustrativa) ──
SERVICO_POR_ACHADO = {
    "RETORNO_ATRASADO": "Agendar o retorno pendente",
    "VACINA_ATRASADA": "Atualizar o protocolo vacinal",
    "VACINA_SEM_REGISTRO": "Atualizar o protocolo vacinal",
    "VARIACAO_PESO": "Consulta para avaliar a variação de peso (exames a critério do veterinário)",
    "SINTOMAS_RECENTES": "Reavaliação clínica dos sintomas",
    "BPM_PERSISTENTE": "Avaliação cardiológica", "BPM_ISOLADO": "Acompanhar os batimentos e reavaliar se repetir",
    "ALERTA_COLEIRA": "Avaliação cardiológica", "CONSULTA_PERDIDA": "Remarcar a consulta perdida",
}

def recomendar_servicos(ctx, score):
    s = []
    for a in score["achados"]:
        sv = SERVICO_POR_ACHADO.get(a["codigo"])
        if sv and sv not in s: s.append(sv)
    p = ctx["pet"]; gato = _norm(p["especie"]) == "gato"
    if p["idade_anos"] >= (10 if gato else 7): s.append("Check-up geriátrico preventivo (exames de sangue e avaliação geral)")
    if not gato and _norm(p["porte"]) == "pequeno" and p["idade_anos"] >= 5: s.append("Avaliação odontológica preventiva")
    return s[:4]

for i in range(1, 5):
    c = montar_contexto(i); s = calcular_score(c)
    print(f"{c['pet']['nome']:5} score={s['score']:3} {s['nivel']:8} {[a['codigo'] for a in s['achados']]}")

## 4. IA Generativa (Ollama) e orquestração
O LLM **não decide**: recebe score, motivos e serviços já calculados e só escreve o texto. Proteções:
1. *System prompt* restritivo (só usa os dados fornecidos, não diagnostica, não dá dose).
2. **Consentimento (LGPD):** se o tutor desativar a IA, nenhum dado vai ao LLM e nada é gerado.
3. **Guarda de emergência** no chat: palavras de urgência nem chegam ao LLM, a resposta é fixa.
4. **Fallback** determinístico se o Ollama estiver fora do ar.
5. Aviso de apoio à decisão anexado pelo código, não pelo modelo.

In [ ]:
SYSTEM_BASE = """Você é o assistente de IA da plataforma SuperVet, usada por tutores e clínicas veterinárias.
Regras obrigatórias:
1. Responda sempre em português do Brasil.
2. Use SOMENTE os dados fornecidos no JSON. Se uma informação não estiver nos dados, diga que não há registro.
3. Você NÃO diagnostica, NÃO prescreve e NÃO sugere doses ou troca de medicamentos. Você é apoio à decisão; quem confirma diagnóstico e tratamento é o veterinário.
4. Não invente números, datas, nomes de medicamentos ou exames.
5. O nível, os motivos e os serviços sugeridos já foram calculados pelo sistema: não recalcule nem contradiga."""

TAREFA_NOTIFICACAO = """Escreva uma notificação curta para o TUTOR do pet (máximo 3 frases), em tom acolhedor e linguagem simples, sem termos técnicos. Cite o motivo mais importante da lista 'motivos' e sugira UMA ação prática, a primeira de 'servicos_sugeridos'. Se não houver motivos, diga que está tudo em dia. Não use listas nem títulos."""

TAREFA_RESUMO = """Gere um resumo pré-consulta para o VETERINÁRIO, em até 180 palavras, com exatamente estas seções em Markdown:
**Perfil**
**O que mudou desde o último atendimento**
**Pontos de atenção (motor de regras)**
**Tratamentos em curso**
**Itens a verificar na consulta**
Em 'Itens a verificar', sugira apenas checagens e perguntas de anamnese, sem diagnosticar."""

TAREFA_CHAT = """Responda à dúvida do TUTOR em até 120 palavras, em linguagem simples. Se a resposta depender de avaliação clínica, oriente a falar com o veterinário. Se a dúvida não puder ser respondida com os dados fornecidos, diga isso."""

DISCLAIMER = "\n\n_Apoio informativo gerado por IA. Não substitui a avaliação do veterinário, que confirma diagnósticos e tratamentos._"

EMERGENCIAS = [r"convuls", r"envenen", r"veneno", r"sangr", r"hemorrag", r"nao (esta )?respir", r"falta de ar", r"engasg", r"desmai",
               r"atropel", r"intoxic", r"parou de respirar", r"nao acorda", r"\bchocolate\b", r"\buvas?\b", r"barriga (muito )?(inchada|dura)"]

def chamar_llm(system, user, temperature=0.2, max_tokens=400):
    r = requests.post(f"{OLLAMA_URL}/api/chat", timeout=300, json={
        "model": MODELO, "stream": False, "keep_alive": "30m",
        "options": {"temperature": temperature, "num_predict": max_tokens},
        "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}]})
    r.raise_for_status()
    return r.json()["message"]["content"].strip()

def resumo_coleira(ctx):
    b = [x["bpm"] for x in ctx["coleira_24h"] if x.get("bpm") is not None]
    if not b: return {"leituras": 0}
    return {"leituras": len(b), "bpm_min": min(b), "bpm_medio": round(sum(b) / len(b)), "bpm_max": max(b),
            "alertas": sum(1 for x in ctx["coleira_24h"] if x.get("alerta"))}

def contexto_para_llm(ctx, score):
    # Versão compacta: nunca manda as 48 leituras cruas, só o resumo
    return {"pet": ctx["pet"],
            "prioridade": {"score": score["score"], "nivel": score["nivel"], "motivos": [a["motivo"] for a in score["achados"]]},
            "servicos_sugeridos": recomendar_servicos(ctx, score),
            "ultimos_prontuarios": ctx["prontuarios"][-3:], "historico_saude": ctx["historico_saude"][-5:],
            "prescricoes_ativas": [p for p in ctx["prescricoes"] if p.get("ativa", True)],
            "coleira_24h_resumo": resumo_coleira(ctx)}

def _prompt(tarefa, ctx, score, extra=""):
    return f"{tarefa}\n\nDADOS (JSON):\n{json.dumps(contexto_para_llm(ctx, score), ensure_ascii=False)}\n{extra}"

def fallback_notificacao(ctx, score):
    nome = ctx["pet"]["nome"]
    if not score["achados"]: return f"{nome} está com tudo em dia! Nenhum ponto de atenção foi identificado agora."
    return f"Atenção com {nome} (prioridade {score['nivel'].lower()}): " + " ".join(a["motivo"] for a in score["achados"][:2]) + " Recomendamos agendar um atendimento com o veterinário."

def fallback_resumo(ctx, score):
    p = ctx["pet"]; ult = ctx["prontuarios"][-1] if ctx["prontuarios"] else None
    return "\n\n".join([
        f"**Perfil**\n{p['nome']}, {p['especie']} {p['raca']} ({p['porte'].lower()}), {p['idade_anos']} anos.",
        "**O que mudou desde o último atendimento**\n" + (f"Último prontuário em {_br(_d(ult['data']))}: {ult['diagnostico']}." if ult else "Sem prontuários."),
        "**Pontos de atenção (motor de regras)**\n" + ("\n".join(f"- {a['motivo']}" for a in score["achados"]) or "Nenhum."),
        "**Tratamentos em curso**\n" + ("\n".join(f"- {r['tipo'].title()}: {r['descricao']}" for r in ctx["prescricoes"] if r.get("ativa", True)) or "Nenhum registrado.")])

def gerar_notificacao_tutor(ctx, score):
    try:
        return chamar_llm(SYSTEM_BASE, _prompt(TAREFA_NOTIFICACAO, ctx, score), max_tokens=200) + DISCLAIMER, "llm"
    except Exception:
        return fallback_notificacao(ctx, score) + DISCLAIMER, "regras (fallback)"

# ── Orquestração: banco → regras → LLM → banco ──
def tem_consentimento(id_pet):
    r = q("SELECT t.CONSENTE_IA AS c FROM TB_TUTOR t JOIN TB_PET p ON p.ID_TUTOR = t.ID_TUTOR WHERE p.ID_PET = ?", id_pet)
    return bool(r and r[0]["c"])

def salvar_consentimento(id_pet, valor):
    exec_sql("UPDATE TB_TUTOR SET CONSENTE_IA = ? WHERE ID_TUTOR = (SELECT ID_TUTOR FROM TB_PET WHERE ID_PET = ?)", int(bool(valor)), id_pet)

def analisar_pet(id_pet):
    if not tem_consentimento(id_pet): return {"id_pet": id_pet, "desativado": True}
    ctx = montar_contexto(id_pet); score = calcular_score(ctx)
    t0 = time.perf_counter()
    texto, origem = gerar_notificacao_tutor(ctx, score)
    dto = {"id_pet": id_pet, "nome_pet": ctx["pet"]["nome"], "score": score["score"], "nivel": score["nivel"], "motivos": score["achados"],
           "servicos": recomendar_servicos(ctx, score), "mensagem_tutor": texto, "origem_texto": origem,
           "latencia_s": round(time.perf_counter() - t0, 2)}
    if score["nivel"] != "BAIXO":   # grava no banco (TB_NOTIFICACAO) para a clínica acompanhar
        exec_sql("INSERT INTO TB_NOTIFICACAO (ID_PET, MENSAGEM, PRIORIDADE, STATUS, CRIADA_EM) VALUES (?,?,?,'PENDENTE',?)",
                 id_pet, texto, score["nivel"], datetime.now().isoformat(timespec="seconds"))
    return dto

def marcar_agendada(id_pet):
    # Retroalimentação: o tutor informa que agendou; a clínica vê na fila
    return exec_sql("UPDATE TB_NOTIFICACAO SET STATUS = 'AGENDADA' WHERE ID_NOTIFICACAO = (SELECT MAX(ID_NOTIFICACAO) FROM TB_NOTIFICACAO WHERE ID_PET = ?)", id_pet) > 0

def resumo_veterinario(id_pet):
    if not tem_consentimento(id_pet): return {"id_pet": id_pet, "desativado": True}
    ctx = montar_contexto(id_pet); score = calcular_score(ctx); t0 = time.perf_counter()
    try:
        texto, origem = chamar_llm(SYSTEM_BASE, _prompt(TAREFA_RESUMO, ctx, score), max_tokens=500), "llm"
    except Exception:
        texto, origem = fallback_resumo(ctx, score), "regras (fallback)"
    return {"id_pet": id_pet, "nome_pet": ctx["pet"]["nome"], "score": score["score"], "nivel": score["nivel"],
            "resumo": texto + DISCLAIMER, "origem_texto": origem, "latencia_s": round(time.perf_counter() - t0, 2)}

def eh_emergencia(pergunta):
    return any(re.search(p, _norm(pergunta)) for p in EMERGENCIAS)

def chat_tutor(id_pet, pergunta):
    if eh_emergencia(pergunta):   # nem chega ao LLM
        return {"id_pet": id_pet, "origem_texto": "regra_emergencia", "escalar_para_clinica": True,
                "resposta": "⚠️ O que você descreveu pode ser uma **emergência**. Procure atendimento veterinário de urgência agora ou ligue para a clínica. Não dê medicamentos por conta própria."}
    if not tem_consentimento(id_pet): return {"id_pet": id_pet, "desativado": True}
    ctx = montar_contexto(id_pet); score = calcular_score(ctx)
    try:
        resp, origem = chamar_llm(SYSTEM_BASE, _prompt(TAREFA_CHAT, ctx, score, f"\nPERGUNTA DO TUTOR: {pergunta}"), max_tokens=300), "llm"
    except Exception:
        resp, origem = f"No momento não consegui consultar o assistente. Para essa dúvida sobre {ctx['pet']['nome']}, fale com o veterinário da clínica.", "regras (fallback)"
    return {"id_pet": id_pet, "resposta": resp + DISCLAIMER, "origem_texto": origem, "escalar_para_clinica": False}

def fila_clinica():
    # Só regras (sem LLM): rápida e auditável
    fila = []
    for r in q("SELECT ID_PET FROM TB_PET"):
        pid = r["ID_PET"]; ctx = montar_contexto(pid)
        ultima = q("SELECT STATUS FROM TB_NOTIFICACAO WHERE ID_PET = ? ORDER BY ID_NOTIFICACAO DESC LIMIT 1", pid)
        item = {"id_pet": pid, "pet": ctx["pet"]["nome"], "ia_ativa": tem_consentimento(pid), "acao_tutor": ultima[0]["STATUS"] if ultima else "—"}
        if item["ia_ativa"]:
            s = calcular_score(ctx); sv = recomendar_servicos(ctx, s)
            item.update(score=s["score"], nivel=s["nivel"], motivos=[a["motivo"] for a in s["achados"][:2]], servico=sv[0] if sv else "—")
        else:
            item.update(score=-1, nivel="IA DESATIVADA", motivos=[], servico="—")
        fila.append(item)
    return sorted(fila, key=lambda x: -x["score"])

## 5. Testes e validação

In [ ]:
from contextlib import contextmanager

@contextmanager
def sem_llm():
    global OLLAMA_URL
    original, OLLAMA_URL = OLLAMA_URL, "http://127.0.0.1:9"   # porta sem serviço
    try: yield
    finally: OLLAMA_URL = original

@contextmanager
def llm_proibido():
    global chamar_llm
    original = chamar_llm
    def explode(*a, **k): raise AssertionError("LLM não deveria ser chamado")
    chamar_llm = explode
    try: yield
    finally: chamar_llm = original

def ctx_base(**kw):
    c = {"id_pet": 99, "referencia_em": AGORA.isoformat(),
         "pet": {"nome": "Teste", "sexo": "M", "especie": "Cachorro", "raca": "SRD", "porte": "GRANDE", "idade_anos": 3},
         "prontuarios": [], "consultas": [], "prescricoes": [], "coleira_24h": [],
         "historico_saude": [{"evento": "VACINA", "data": passado(10), "detalhes": ""}]}
    c.update(kw); return c

def codigos(ctx): return {a["codigo"]: a["pontos"] for a in calcular_score(ctx)["achados"]}
def leituras(bpms, alerta=None): return [{"datahora": AGORA.isoformat(), "bpm": b, "alerta": alerta} for b in bpms]

# ── regras ──
def test_pet_saudavel_tem_score_zero():
    assert calcular_score(ctx_base())["score"] == 0

def test_vacina_atrasada_e_sem_registro():
    assert codigos(ctx_base(historico_saude=[{"evento": "VACINA", "data": passado(400), "detalhes": ""}])) == {"VACINA_ATRASADA": 20}
    assert codigos(ctx_base(historico_saude=[])) == {"VACINA_SEM_REGISTRO": 20}

def test_variacao_de_peso():
    p = [{"data": passado(60), "peso_kg": 10.0, "sintomas": ""}, {"data": passado(30), "peso_kg": 8.5, "sintomas": ""}]
    assert codigos(ctx_base(prontuarios=p)) == {"VARIACAO_PESO": 20}
    p[1]["peso_kg"] = 10.6
    assert codigos(ctx_base(prontuarios=p)) == {"VARIACAO_PESO": 10}

def test_retorno_atrasado_e_cumprido():
    p = [{"data": passado(30), "peso_kg": None, "sintomas": "", "data_retorno": passado(5)}]
    assert codigos(ctx_base(prontuarios=p)) == {"RETORNO_ATRASADO": 15}
    assert codigos(ctx_base(prontuarios=p, consultas=[{"data": passado(2), "status": "REALIZADA"}])) == {}

def test_bpm_respeita_o_porte():
    assert codigos(ctx_base(coleira_24h=leituras([130]))) == {"BPM_ISOLADO": 10}
    assert codigos(ctx_base(coleira_24h=leituras([130, 131, 132]))) == {"BPM_PERSISTENTE": 25}
    pequeno = ctx_base(coleira_24h=leituras([130, 131, 132])); pequeno["pet"]["porte"] = "PEQUENO"
    assert codigos(pequeno) == {}

def test_alerta_da_coleira():
    assert codigos(ctx_base(coleira_24h=leituras([80], alerta="TAQUICARDIA"))) == {"ALERTA_COLEIRA": 20}

def test_sintoma_recorrente():
    p = [{"data": passado(100), "sintomas": "Vômito", "peso_kg": None}, {"data": passado(10), "sintomas": "vômito, apatia", "peso_kg": None}]
    assert codigos(ctx_base(prontuarios=p)) == {"SINTOMAS_RECENTES": 10, "SINTOMA_RECORRENTE": 10}

# ── banco + fila ──
def test_cenarios_do_banco_e_ordenacao_da_fila():
    esperado = {"Thor": ("BAIXO", 0), "Mel": ("MEDIO", 45), "Luna": ("ALTO", 55), "Rex": ("CRITICO", 80)}
    fila = fila_clinica()
    for f in fila: assert (f["nivel"], f["score"]) == esperado[f["pet"]], f
    assert [f["pet"] for f in fila] == ["Rex", "Luna", "Mel", "Thor"]

def test_servicos_sugeridos():
    rex = montar_contexto(4); thor = montar_contexto(1)
    assert any("cardiol" in s.lower() for s in recomendar_servicos(rex, calcular_score(rex)))
    assert recomendar_servicos(thor, calcular_score(thor)) == []

# ── privacidade, consentimento, segurança ──
def test_nenhum_dado_pessoal_no_contexto_nem_no_prompt():
    for pid in range(1, 5):
        c = montar_contexto(pid)
        blob = (json.dumps(c, ensure_ascii=False) + _prompt(TAREFA_NOTIFICACAO, c, calcular_score(c))).lower()
        for proibido in ["@", "cpf", "tutor exemplo", "(11)", "latitude", "longitude"]:
            assert proibido not in blob, (pid, proibido)

def test_consentimento_desativa_a_ia():
    salvar_consentimento(1, False)
    try:
        with llm_proibido():
            assert analisar_pet(1).get("desativado") and resumo_veterinario(1).get("desativado") and chat_tutor(1, "Posso passear hoje?").get("desativado")
        assert next(f for f in fila_clinica() if f["id_pet"] == 1)["ia_ativa"] is False
    finally:
        salvar_consentimento(1, True)

def test_emergencia_nao_chama_o_llm():
    with llm_proibido():
        r = chat_tutor(1, "Meu cachorro teve uma convulsão agora!")
    assert r["origem_texto"] == "regra_emergencia" and r["escalar_para_clinica"] is True
    assert eh_emergencia("ele comeu CHOCOLATE") and not eh_emergencia("qual a melhor hora para passear?")

def test_fallback_quando_llm_esta_fora():
    with sem_llm():
        d = analisar_pet(4)
    assert d["origem_texto"] == "regras (fallback)" and d["nivel"] == "CRITICO" and "Rex" in d["mensagem_tutor"]

def test_retroalimentacao_do_tutor_chega_na_clinica():
    with sem_llm(): analisar_pet(4)
    assert next(f for f in fila_clinica() if f["id_pet"] == 4)["acao_tutor"] == "PENDENTE"
    assert marcar_agendada(4) is True
    assert next(f for f in fila_clinica() if f["id_pet"] == 4)["acao_tutor"] == "AGENDADA"

exec_sql("DELETE FROM TB_NOTIFICACAO")
falhas = 0
for nome, fn in sorted((n, f) for n, f in globals().items() if n.startswith("test_") and callable(f)):
    try: fn(); print("✅", nome)
    except Exception as e: falhas += 1; print("❌", nome, "->", repr(e))
exec_sql("DELETE FROM TB_NOTIFICACAO")   # limpa as notificações geradas pelos testes
print(f"\n{'Todos os testes passaram.' if not falhas else f'{falhas} teste(s) falharam.'}")

## 6. Avaliação dos resultados e evidências
Roda a IA nos 4 pets e mede: (1) score obtido × esperado, (2) **latência** do LLM, (3) **fidelidade**: números citados no texto que não aparecem nos dados enviados (indicador simples de alucinação) e (4) origem do texto (`llm` ou fallback).
Gera o arquivo `evidencias_ia.json` para o repositório e deixa notificações no banco para a demonstração.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

def numeros(txt): return set(re.findall(r"\d+(?:\.\d+)?", txt.replace(",", ".")))

ESPERADO = {"Thor": ("BAIXO", 0), "Mel": ("MEDIO", 45), "Luna": ("ALTO", 55), "Rex": ("CRITICO", 80)}
linhas, notificacoes = [], {}
for pid in range(1, 5):
    d = analisar_pet(pid); ctx = montar_contexto(pid)
    dados = json.dumps(contexto_para_llm(ctx, calcular_score(ctx)), ensure_ascii=False)
    estranhos = sorted(numeros(d["mensagem_tutor"]) - numeros(dados)) if d["origem_texto"] == "llm" else []
    esp = ESPERADO[d["nome_pet"]]
    linhas.append({"pet": d["nome_pet"], "score": d["score"], "nivel": d["nivel"], "esperado": f"{esp[0]} ({esp[1]})",
                   "bate_com_esperado": (d["nivel"], d["score"]) == esp, "texto_gerado_por": d["origem_texto"],
                   "latencia_s": d["latencia_s"], "numeros_fora_dos_dados": estranhos})
    notificacoes[d["nome_pet"]] = d["mensagem_tutor"]
avaliacao = pd.DataFrame(linhas)
display(avaliacao)

resumo_luna = resumo_veterinario(3)
chats = [{"pergunta": p, **chat_tutor(i, p)} for i, p in [(2, "A Mel pode comer petisco hoje? Quando ela precisa voltar ao veterinário?"),
                                                            (1, "O Thor engoliu chocolate agora há pouco, o que eu faço?")]]
display(Markdown(f"**Resumo da Luna** (texto: {resumo_luna['origem_texto']}, {resumo_luna['latencia_s']}s)\n\n{resumo_luna['resumo']}"))
for c in chats: display(Markdown(f"**Tutor:** {c['pergunta']}\n\n**IA** _({c['origem_texto']})_: {c['resposta']}"))

with open("evidencias_ia.json", "w", encoding="utf-8") as f:
    json.dump({"gerado_em": datetime.now().isoformat(timespec="seconds"), "modelo": MODELO, "avaliacao": linhas,
               "notificacoes_tutor": notificacoes, "resumo_veterinario_luna": resumo_luna, "chat_tutor": chats,
               "tb_notificacao": q("SELECT * FROM TB_NOTIFICACAO")}, f, ensure_ascii=False, indent=2)
try:
    from google.colab import files; files.download("evidencias_ia.json")
except Exception:
    print("Salvo em evidencias_ia.json")

## 7. Aplicação (Gradio): três abas, uma por persona
**Tutor** vê o que precisa de atenção e conversa com o assistente · **Veterinário** recebe o resumo pré-consulta · **Clínica** vê a fila priorizada.
Use o **link público** que aparece abaixo da célula (funciona enquanto o notebook estiver rodando).

In [ ]:
import gradio as gr

EMOJI = {"BAIXO": "🟢", "MEDIO": "🟡", "ALTO": "🟠", "CRITICO": "🔴"}
MSG_DESATIVADO = "🔒 Os recursos de IA estão **desativados** para este tutor. Marque a opção de consentimento acima para ativar."

def opcoes_pets():
    return [(f"{r['NOME']} ({r['ESPECIE']}, {r['RACA']})", r["ID_PET"]) for r in q(
        "SELECT p.ID_PET, p.NOME, e.NOME AS ESPECIE, r.NOME AS RACA FROM TB_PET p JOIN TB_RACA r ON r.ID_RACA = p.ID_RACA JOIN TB_ESPECIE e ON e.ID_ESPECIE = r.ID_ESPECIE ORDER BY p.ID_PET")]

def ui_pet(id_pet):
    p = montar_contexto(int(id_pet))["pet"]
    return tem_consentimento(int(id_pet)), f"**{p['nome']}** · {p['especie']} {p['raca']} · porte {p['porte'].lower()} · {p['idade_anos']} anos"

def ui_consentimento(id_pet, valor):
    salvar_consentimento(int(id_pet), valor)

def ui_tutor(id_pet):
    d = analisar_pet(int(id_pet))
    if d.get("desativado"): return MSG_DESATIVADO
    md_ = [f"### {EMOJI[d['nivel']]} {d['nome_pet']}: prioridade {d['nivel']} ({d['score']}/100)"]
    if d["motivos"]: md_.append("**Por que:**\n" + "\n".join(f"- {m['motivo']} _(+{m['pontos']})_" for m in d["motivos"]))
    md_.append("**Mensagem para você:**\n\n" + d["mensagem_tutor"])
    md_.append("**Serviços sugeridos:**\n" + ("\n".join(f"- {s}" for s in d["servicos"]) if d["servicos"] else "- Nenhum por enquanto. Está tudo em dia!"))
    md_.append(f"_Texto gerado por: {d['origem_texto']} · {d['latencia_s']}s_")
    return "\n\n".join(md_)

def ui_agendar(id_pet):
    return "✅ Obrigado! A clínica foi avisada de que você agendou." if marcar_agendada(int(id_pet)) else "Não há notificação pendente para este pet. Clique antes em “Ver o que precisa de atenção”."

def ui_chat(id_pet, pergunta):
    if not pergunta.strip(): return "Digite uma pergunta."
    r = chat_tutor(int(id_pet), pergunta)
    if r.get("desativado"): return MSG_DESATIVADO
    return f"{r['resposta']}\n\n_origem: {r['origem_texto']}_"

def ui_resumo(id_pet):
    r = resumo_veterinario(int(id_pet))
    if r.get("desativado"): return MSG_DESATIVADO
    return f"### {EMOJI[r['nivel']]} {r['nome_pet']} · {r['nivel']} ({r['score']}/100)\n\n{r['resumo']}\n\n_texto gerado por: {r['origem_texto']} · {r['latencia_s']}s_"

def ui_fila():
    return [[f"{EMOJI.get(f['nivel'], '🔒')} {f['pet']}", f["nivel"], "—" if f["score"] < 0 else f["score"],
             " | ".join(f["motivos"]) or "—", f["servico"], f["acao_tutor"]] for f in fila_clinica()]

with gr.Blocks(title="SuperVet IA") as demo:
    gr.Markdown("# 🐾 SuperVet IA\nPrioridades e recomendações de saúde do pet: **regras decidem, IA explica**.")
    with gr.Row():
        dd = gr.Dropdown(opcoes_pets(), value=1, label="Escolha o pet", scale=2)
        consent = gr.Checkbox(value=True, label="Autorizo o uso de IA com os dados do meu pet (LGPD)", scale=3)
    info = gr.Markdown()
    with gr.Tabs():
        with gr.Tab("🐾 Tutor"):
            b_tutor = gr.Button("Ver o que precisa de atenção", variant="primary")
            s_tutor = gr.Markdown()
            b_agendar = gr.Button("✅ Já agendei a consulta")
            s_agendar = gr.Markdown()
            gr.Markdown("---\n#### Pergunte ao assistente")
            t_pergunta = gr.Textbox(label="Sua dúvida", placeholder="Ex.: Posso dar petisco hoje? Quando devo voltar ao veterinário?")
            b_chat = gr.Button("Perguntar")
            s_chat = gr.Markdown()
        with gr.Tab("🩺 Veterinário"):
            b_vet = gr.Button("Gerar resumo pré-consulta", variant="primary")
            s_vet = gr.Markdown()
        with gr.Tab("🏥 Clínica"):
            b_fila = gr.Button("Atualizar fila de acompanhamento", variant="primary")
            tabela = gr.Dataframe(headers=["Pet", "Prioridade", "Score", "Principais motivos", "Serviço sugerido", "Ação do tutor"], interactive=False, wrap=True)

    dd.change(ui_pet, dd, [consent, info])
    consent.change(ui_consentimento, [dd, consent], None)
    b_tutor.click(ui_tutor, dd, s_tutor)
    b_agendar.click(ui_agendar, dd, s_agendar)
    b_chat.click(ui_chat, [dd, t_pergunta], s_chat)
    t_pergunta.submit(ui_chat, [dd, t_pergunta], s_chat)
    b_vet.click(ui_resumo, dd, s_vet)
    b_fila.click(ui_fila, None, tabela)
    demo.load(ui_pet, dd, [consent, info])
    demo.load(ui_fila, None, tabela)

demo.launch(share=True)